# App2: Reaction Load Params - REST API to JSON Schema

**Workspace ID:** 2673  
**Entity ID:** 12163  
**App URL:** https://demo.viktor.ai/workspaces/2673/app/editor/12163

This notebook reads the entity data through the VIKTOR REST API and converts the saved parameters/properties to JSON Schema.

In [ ]:
import os
import json
from pathlib import Path
from typing import Any

import requests
from dotenv import find_dotenv, load_dotenv

DOTENV_PATH = find_dotenv(usecwd=True)
load_dotenv(DOTENV_PATH, override=True)

print(f"Loaded .env: {DOTENV_PATH or '<not found>'}")


def get_env_int(*names: str, default: int) -> int:
    for name in names:
        raw_value = os.getenv(name)
        if raw_value and raw_value.strip():
            try:
                return int(raw_value)
            except ValueError as exc:
                raise ValueError(f"{name} must be an integer.") from exc
    return default


WORKSPACE_ID = get_env_int("APP2_WORKSPACE_ID", "VIKTOR_WORKSPACE_ID", default=2673)
ENTITY_ID = get_env_int("APP2_ENTITY_ID", "VIKTOR_ENTITY_ID", default=12163)
OUTPUT_FILE = os.getenv("APP2_OUTPUT_FILE", "app2_params_schema.json")

print(f"Target: Workspace {WORKSPACE_ID}, Entity {ENTITY_ID}")

## 1. Connect to VIKTOR REST API

The REST helper below follows the repository VIKTOR REST API rules: normalize the API base once, read the token from environment variables, do not print the token, centralize timeouts and errors, and pass query parameters through `params=...`.

In [ ]:
# Initialize a small REST API client with a Personal Access Token.
# Supported token environment variables, in order:
# TOKEN_VK_APP, VIKTOR_TOKEN, VIKTOR_API_TOKEN.

TOKEN_ENV_NAMES = ("TOKEN_VK_APP", "VIKTOR_TOKEN", "VIKTOR_API_TOKEN")


def normalize_bearer_token(raw_token: str, *, env_var: str) -> str:
    token = raw_token.strip().strip('"').strip("'").strip()

    if token.startswith(f"{env_var}="):
        token = token.split("=", 1)[1].strip().strip('"').strip("'").strip()

    if token.lower().startswith("authorization:"):
        token = token.split(":", 1)[1].strip()

    if token.lower().startswith("bearer "):
        token = token.split(None, 1)[1].strip()

    token = token.strip().strip('"').strip("'").strip()
    if not token:
        raise ValueError(f"{env_var} is empty.")
    if any(ch.isspace() for ch in token):
        raise ValueError(f"{env_var} contains whitespace. Paste only the token value.")
    return token


def get_token() -> tuple[str, str]:
    for env_var in TOKEN_ENV_NAMES:
        raw_token = os.getenv(env_var)
        if raw_token and raw_token.strip():
            return normalize_bearer_token(raw_token, env_var=env_var), env_var
    raise ValueError(
        "Missing VIKTOR token. Set TOKEN_VK_APP or VIKTOR_TOKEN. "
        "VIKTOR_API_TOKEN is also accepted for backwards compatibility."
    )


def get_api_base() -> str:
    api_base = os.getenv("VIKTOR_API_BASE")
    if api_base and api_base.strip():
        base = api_base.strip().rstrip("/")
    else:
        environment = os.getenv("VIKTOR_ENVIRONMENT", "demo").strip().rstrip("/")
        if not environment:
            raise ValueError("Missing VIKTOR_ENVIRONMENT.")
        if environment.startswith("https://"):
            base = environment
        elif environment.startswith("http://"):
            base = "https://" + environment.removeprefix("http://")
        elif environment.endswith(".viktor.ai"):
            base = f"https://{environment}"
        else:
            base = f"https://{environment}.viktor.ai"

    base = base.rstrip("/")
    if base.startswith("http://"):
        base = "https://" + base.removeprefix("http://")
    if not base.startswith("https://"):
        raise ValueError("VIKTOR API base must resolve to an HTTPS URL.")
    return base if base.endswith("/api") else f"{base}/api"


class ViktorRestClient:
    def __init__(
        self,
        *,
        api_base: str,
        token: str,
        connect_timeout: float = 5.0,
        read_timeout: float = 30.0,
    ) -> None:
        token = token.strip()
        if not token:
            raise ValueError("Missing VIKTOR token.")

        base = api_base.strip().rstrip("/")
        self.api_base = base if base.endswith("/api") else f"{base}/api"
        self.timeout = (connect_timeout, read_timeout)
        self.session = requests.Session()
        self.session.headers.update(
            {
                "Authorization": f"Bearer {token}",
                "Accept": "application/json",
            }
        )

    def url(self, path_or_url: str) -> str:
        if path_or_url.startswith("http://") or path_or_url.startswith("https://"):
            return path_or_url
        return f"{self.api_base}/{path_or_url.lstrip('/')}"

    def check_response(self, response: requests.Response, *, action: str) -> None:
        if response.ok:
            return
        body = response.text[:500]
        raise RuntimeError(f"{action} failed (status={response.status_code}): {body}")

    def request_json(
        self,
        method: str,
        path_or_url: str,
        *,
        params: dict[str, Any] | None = None,
        json_body: dict[str, Any] | None = None,
        action: str = "VIKTOR REST request",
    ) -> dict[str, Any]:
        response = self.session.request(
            method=method.upper(),
            url=self.url(path_or_url),
            params=params,
            json=json_body,
            timeout=self.timeout,
        )
        self.check_response(response, action=action)
        if not response.content:
            return {}
        try:
            return response.json()
        except ValueError as exc:
            raise RuntimeError(
                f"{action} did not return valid JSON: {response.text[:500]}"
            ) from exc

    def get_json(
        self,
        path_or_url: str,
        *,
        params: dict[str, Any] | None = None,
        action: str = "GET request",
    ) -> dict[str, Any]:
        return self.request_json("GET", path_or_url, params=params, action=action)

    def build_entity_editor_url(self, *, workspace_id: int, entity_id: int) -> str:
        ui_base = self.api_base[:-4] if self.api_base.endswith("/api") else self.api_base
        return f"{ui_base}/workspaces/{workspace_id}/app/editor/{entity_id}"


api_token, token_env_var = get_token()
client = ViktorRestClient(api_base=get_api_base(), token=api_token)

print(f"Using VIKTOR token from environment variable: {token_env_var}")
print(f"REST API base URL: {client.api_base}")

workspaces = client.get_json(
    "workspaces/",
    params={"limit": 1, "offset": 0, "detail_level": "minimal"},
    action="List workspaces",
)
if "results" not in workspaces:
    raise RuntimeError(f"Unexpected workspace list response keys: {list(workspaces.keys())}")

print("Connected to VIKTOR REST API")
print(f"Workspace list response keys: {list(workspaces.keys())}")

## 2. Fetch Entity and Parametrization

In [ ]:
# Fetch the entity through REST.
entity = client.get_json(
    f"workspaces/{WORKSPACE_ID}/entities/{ENTITY_ID}/",
    params={
        "properties": "true",
        "clean_params": "true",
        "param_types": "true",
    },
    action="Get entity",
)

print(f"Entity: {entity.get('name', '<no name returned>')}")
print(f"Type: {entity.get('entity_type_name', entity.get('entity_type', '<no type returned>'))}")
print(f"Editor URL: {client.build_entity_editor_url(workspace_id=WORKSPACE_ID, entity_id=ENTITY_ID)}")
print(f"Response keys: {list(entity.keys())}")


def extract_saved_params(entity_payload: dict[str, Any]) -> dict[str, Any]:
    """Extract saved params/properties from a VIKTOR REST entity response."""
    for key in ("properties", "params", "last_saved_params"):
        value = entity_payload.get(key)
        if isinstance(value, dict):
            return value

    for key in ("last_saved_revision", "latest_revision", "revision"):
        value = entity_payload.get(key)
        if isinstance(value, dict) and isinstance(value.get("params"), dict):
            return value["params"]

    raise KeyError(
        "No saved params/properties found in the REST response. "
        f"Available keys: {list(entity_payload.keys())}"
    )


params = extract_saved_params(entity)
print(f"\nParameter keys: {list(params.keys()) if params else 'None'}")

## 3. Get Parametrization Definition

We'll extract the parametrization structure by analyzing the entity type and saved parameters.

In [ ]:
def infer_json_schema_from_params(params: dict[str, Any]) -> dict[str, Any]:
    schema: dict[str, Any] = {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    }

    for key, value in params.items():
        schema["properties"][key] = infer_type_from_value(value)
        if value is not None:
            schema["required"].append(key)

    if not schema["required"]:
        schema.pop("required")
    return schema


def infer_type_from_value(value: Any) -> dict[str, Any]:
    if isinstance(value, bool):
        return {"type": "boolean"}
    if isinstance(value, int) and not isinstance(value, bool):
        return {"type": "integer"}
    if isinstance(value, float):
        return {"type": "number"}
    if isinstance(value, str):
        return {"type": "string"}
    if isinstance(value, list):
        return {
            "type": "array",
            "items": infer_array_item_schema(value),
        }
    if isinstance(value, dict):
        return infer_json_schema_from_params(value)
    if value is None:
        return {"type": ["string", "null"]}
    return {"type": "string"}


def infer_array_item_schema(values: list[Any]) -> dict[str, Any]:
    if not values:
        return {"type": "string"}
    if all(isinstance(item, dict) for item in values):
        merged: dict[str, Any] = {}
        required_candidates: set[str] | None = None
        for item in values:
            item_schema = infer_json_schema_from_params(item)
            merged.update(item_schema.get("properties", {}))
            item_required = set(item_schema.get("required", []))
            required_candidates = (
                item_required
                if required_candidates is None
                else required_candidates.intersection(item_required)
            )
        schema: dict[str, Any] = {
            "type": "object",
            "properties": merged,
            "additionalProperties": False,
        }
        if required_candidates:
            schema["required"] = sorted(required_candidates)
        return schema
    return infer_type_from_value(values[0])


if params:
    schema = infer_json_schema_from_params(params)
    print("Schema generated")
    print(f"\nProperties: {list(schema['properties'].keys())}")
else:
    print("No parameters found; creating an empty schema")
    schema = {
        "type": "object",
        "properties": {},
        "additionalProperties": False,
    }

## 4. Preview the Schema

In [ ]:
print(json.dumps(schema, indent=2))

## 5. Save to JSON File

In [ ]:
output_path = Path(OUTPUT_FILE)
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2)

print(f"Schema saved to: {output_path.absolute()}")

## 6. Display Sample Parameters

In [ ]:
print("Sample parameters from entity:")
print(json.dumps(params, indent=2, default=str))